I2 portable reader candidate. New outputs are isolated under runs/. Constructed examples demonstrate methods; fixed filing facts/figures are historical teaching inputs. This does not reproduce a market strategy.

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch11_returns_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# 第11章：收益、复利与路径风险

构造案例解释公式；真实案例读取冻结 French 月度行业组合 NoDur 与同月 RF。频率为月，月份是收益区间标签，不能充当信号公布或成交时间。当前修订版本用于描述性分析，不能证明历史可交易策略。

In [ ]:
from pathlib import Path
import json
import numpy as np
from quant_course.market_data import load_monthly_snapshot
from quant_course.performance import (wealth_index, geometric_mean_return, cagr,
    drawdown_path, annualized_volatility, sharpe_ratio)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "请从课程根目录或 notebooks 目录运行"


## 构造案例：百分数相加会丢掉复利

100先涨50%再跌50%，财富剩75。算术平均为零，几何平均为负；观察初始财富在回撤中的作用。

In [ ]:
returns = np.array([0.5, -0.5])
wealth = wealth_index(returns, initial_capital=100)
assert np.allclose(wealth, [100, 150, 75])
print({"wealth": wealth, "arithmetic_mean": returns.mean(),
       "geometric_mean": geometric_mean_return(returns),
       "drawdown": drawdown_path(wealth)})


## 同终值不同风险

下面两条路径使用相同三期收益。每期设为一年，CAGR相同，最大回撤分别为52%和40%。

In [ ]:
for sequence in ([0.5, -0.4, -0.2], [-0.4, 0.5, -0.2]):
    values = wealth_index(np.array(sequence), 100)
    print(values, cagr(100, float(values[-1]), 3), -drawdown_path(values).min())


## 真实快照：先验字节，再对齐月份

处理结果已是小数收益，不再除100。不补缺失月份，不把 RF 无声填零；120个月对应2010年1月至2019年12月。

French historical empirical branch SKIPPED: G01 unknown; G05 A10-B unauthorized. No market data bundled. See ../README.md and OPTIONAL_DATA.md. Original calculation retained in optional source cell below; not executed in default mode.

```python
industry = json.loads((ROOT / "data/manifests/french_industries_2026-10-05.json").read_text())
factor = json.loads((ROOT / "data/manifests/french_factors_2026-10-05.json").read_text())["datasets"]["ff3"]
asset = load_monthly_snapshot(ROOT / industry["processed_path"], industry["processed_sha256"], ["NoDur"])
rf = load_monthly_snapshot(ROOT / factor["processed_path"], factor["processed_sha256"], ["RF"])
sample = asset.loc["2010-01":"2019-12"].join(rf, how="left", validate="one_to_one")
assert len(sample) == 120 and not sample.isna().any().any()
assert (sample.to_numpy() > -1).all()
r = sample["NoDur"].to_numpy()
risk_free = sample["RF"].to_numpy()
values = wealth_index(r)
metrics = {"final_wealth": float(values[-1]), "CAGR": cagr(1, float(values[-1]), 10),
    "volatility_sqrt12_convention": annualized_volatility(r, 12),
    "Sharpe_sqrt12_convention": sharpe_ratio(r, risk_free, 12),
    "maximum_drawdown_positive_loss": -float(drawdown_path(values).min())}
metrics

```

## 图形与解释

第一幅横轴为月份标签，纵轴是初始1的财富指数，不是价格。第二幅横轴为小数月收益，纵轴为月份数。第三幅使用过去12个月样本标准差乘√12，这是报告约定，不保证精确年度复合风险。第四幅显示相对既往财富峰值的负回撤。绘图调用章节入口的显示辅助函数；全部核心指标和数据校验调用 src 包。

French historical empirical branch SKIPPED: G01 unknown; G05 A10-B unauthorized. No market data bundled. See ../README.md and OPTIONAL_DATA.md. Original calculation retained in optional source cell below; not executed in default mode.

```python
import runpy
from IPython.display import Image, display
plot_helpers = runpy.run_path(str(ROOT / "code/ch11_returns.py"), run_name="plot_helpers")
figure = RUN_DIR / "figures/a3/ch11_returns_dashboard.png"
plot_helpers["make_figure"](sample, figure)
display(Image(filename=str(figure)))

```

## 自检与研究边界

验收同一份实证输出的数字。这里没有交易规则、交易成本、入场时点或样本外预测，因此不能称为Alpha回测。真实收益来自当前发布行业组合快照，不是逐股PIT数据库。

French historical empirical branch SKIPPED: G01 unknown; G05 A10-B unauthorized. No market data bundled. See ../README.md and OPTIONAL_DATA.md. Original calculation retained in optional source cell below; not executed in default mode.

```python
recorded = json.loads((ROOT / "qa/a3/ch11_actual_metrics.json").read_text())
assert np.isclose(metrics["final_wealth"], recorded["wealth_final"])
assert np.isclose(metrics["CAGR"], recorded["cagr_month_period_convention"])
assert np.isclose(metrics["Sharpe_sqrt12_convention"], recorded["sharpe_sqrt12_convention"])
print("同一快照指标一致；不构成策略收益证据。")

```